# Collie

This example shows how to evaluate a `genlm.control` model on the Collie domain.

* **Task**: Generate text that satisfies various constraints (e.g., word count, character count, required words/phrases, paragraph structure).
* **Data**: Collie dataset (Yao et al., 2023) from the official [Princeton-NLP repository](https://github.com/princeton-nlp/Collie).
* **Paper**: [COLLIE: Systematic Construction of Constrained Text Generation Tasks](https://arxiv.org/pdf/2307.08689)

## Setup

First, install the dependencies for this domain. In the root directory, run:    

```bash
pip install -e .[collie]
```

This will install:
- `dill` - for loading the official Collie data format
- `collie-bench` - the official Collie constraint checking library

## Usage 

This example shows how to evaluate a `genlm.control` model on the Collie constrained text generation domain using the official Princeton-NLP dataset.


### Initialize the dataset and evaluator


In [ ]:
from genlm.eval.domains.collie import (
    CollieDataset,
    CollieEvaluator,
    RECOMMENDED_CONSTRAINT_TYPES,
)

In [2]:
dataset = CollieDataset.from_official(
    constraint_types=RECOMMENDED_CONSTRAINT_TYPES,  # Simple count constraints (word, char, sentence)
    max_example_length=300,  # Filter to shorter examples for efficiency
    max_prompt_length=300,  # Filter to shorter prompts
    max_instances=20,
    shuffle=True,
    seed=42,
)

print(f"Instances loaded: {len(dataset)}")
print(f"Constraint types: {RECOMMENDED_CONSTRAINT_TYPES[:3]}...")  # Show first 3
evaluator = CollieEvaluator()  # Requires collie-bench to be installed

Instances loaded: 20
Constraint types: ['wiki_c01', 'guten_c01', 'ccnews_c01']...


### Understanding Dataset Filtering

The Collie dataset contains many constraint types. For AWRS with `CollieConstraintPotential`, we filter to **simple count-based constraints** that work best with guided generation:

** Recommended Constraint Types** (work well with potentials):
- **Word count** (`wiki_c01`, `guten_c01`, `ccnews_c01`): "Generate exactly N words"
- **Character count** (`wiki_c04`, `guten_c04`, `ccnews_c04`): "Generate exactly N characters"
- **Sentence count** (`wiki_c11`, `guten_c11`, `ccnews_c11`): "Generate exactly N sentences"

** Complex Constraint Types** (may not benefit as much):
- Multi-level constraints (e.g., `wiki_c05`): "Each word must have exactly N characters"
- Position-based constraints (e.g., `wiki_c14`): "Paragraph must end with specific sentence"

We also filter by `max_example_length` and `max_prompt_length` to keep examples manageable for demonstration.


In [ ]:
# show dataset
for instance in dataset:
    print(instance.constraint_type)
    print(instance.prompt)
    print(instance.targets)
    print(instance.constraint)
    break

### Guided Generation with AWRS and Constraint Potentials

We'll use AWRS (Adaptive Weighted Rejection Sampling) with the `CollieConstraintPotential` to guide generation toward satisfying constraints during generation (not just after). This dramatically improves constraint satisfaction rates.


In [ ]:
from genlm.control import PromptedLLM, AWRS
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.collie import (
    CollieConstraintPotential,
    default_prompt_formatter,
)

# Load an LLM
LLM = PromptedLLM.from_name(
    "meta-llama/Meta-Llama-3-8B",
    eos_tokens=[b"\n", b"\n\n", b"<|end_of_text|>", b"<|eot_id|>"],
)
# LLM = PromptedLLM.from_name("gpt2", eos_tokens=[b"\n", b"\n\n", b"<|endoftext|>"])


async def model(instance, output_dir, replicate):
    # Set the prompt for the LLM.
    LLM.prompt_ids = default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=False
    )

    # Construct goal validation potential.
    potential = CollieConstraintPotential(
        constraint=instance.constraint,
        targets=instance.targets,
        tolerance=1.5,
        verbose=True,
    ).coerce(LLM, f=b"".join)

    # Define an adaptive weighted rejection sampler to sample tokens from the constrained model.
    sampler = AWRS(LLM, potential)

    # Run SMC to sample sequences from the constrained model.
    sequences = await sampler.smc(
        n_particles=10,
        ess_threshold=0.9,
        max_tokens=150,
    )

    return ModelOutput(
        responses=[
            ModelResponse(response=sequence, weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=20,
    n_replicates=1,
    verbosity=1,
    output_dir="collie_results",  # optionally save the results to a directory
)

## Citation

If you use the Collie dataset in your work, please cite:

Shunyu Yao, Howard Chen, Austin W. Hanjie, Runzhe Yang, and Karthik Narasimhan. COLLIE: Systematic Construction of Constrained Text Generation Tasks. arXiv preprint arXiv:2307.08689, 2023. URL https://arxiv.org/abs/2307.08689
